In [1]:
# Instalasi Unsloth dan library pendukungnya
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-k2_ug0pp/unsloth_afe802db6fb64d30bf1e2edcbf51bbbd
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-k2_ug0pp/unsloth_afe802db6fb64d30bf1e2edcbf51bbbd
  Resolved https://github.com/unslothai/unsloth.git to commit 41b93a9169ab72947f53ed2b43acab422a2643df
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.1 MB/s eta 0:00:00
   ━━

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Otomatis mendeteksi dukungan hardware
load_in_4bit = True # Memampatkan model agar muat di GPU Colab

# Kita gunakan versi Qwen2.5-7B yang sudah disiapkan Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Menambahkan adapter LoRA (bagian otak yang akan dilatih SOP pabrik)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.8.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from datasets import load_dataset

# Format standar Alpaca yang dikenali oleh banyak LLM
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Menggabungkan ketiganya menjadi satu teks panjang
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Memuat dataset buatan AI Agent kamu
dataset = load_dataset("json", data_files="sop_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/140 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported # <-- Tambahkan import fungsi ini

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 50, # Jika loss masih tinggi, angka ini bisa dinaikkan ke 100 atau 150
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(), # <-- Bagian yang diperbaiki
        bf16 = is_bfloat16_supported(),     # <-- Bagian yang diperbaiki
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Memulai proses belajar!
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/140 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 140 | Num Epochs = 3 | Total steps = 50
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.985574
2,1.948734
3,2.006098
4,1.829121
5,1.773435
6,1.714459
7,1.516999
8,1.564605
9,1.513554
10,1.300868


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-50/tokenizer_config.json.


In [ ]:
# Kita hanya menyimpan bagian otak tambahannya (LoRA adapter), bukan seluruh model besar
model.save_pretrained("qwen-sop-model")
tokenizer.save_pretrained("qwen-sop-model")
print("Model berhasil disimpan di folder 'qwen-sop-model'!")

Unsloth: Restored added_tokens_decoder metadata in qwen-sop-model/tokenizer_config.json.


Model berhasil disimpan di folder 'qwen-sop-model'!


In [ ]:
!zip -r qwen-sop-model.zip qwen-sop-model/

  adding: qwen-sop-model/ (stored 0%)
  adding: qwen-sop-model/README.md (deflated 65%)
  adding: qwen-sop-model/tokenizer_config.json (deflated 89%)
  adding: qwen-sop-model/adapter_model.safetensors (deflated 8%)
  adding: qwen-sop-model/tokenizer.json (deflated 81%)
  adding: qwen-sop-model/adapter_config.json (deflated 59%)
  adding: qwen-sop-model/chat_template.jinja (deflated 71%)


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/qwen-sop-model.zip'
extract_path = '/content/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Berhasil di-unzip!")
print("Isi dataset:")
print(os.listdir(extract_path))

Berhasil di-unzip!
Isi dataset:
['.config', 'qwen-sop-model', 'drive', 'sample_data']


In [5]:
!pip install gradio

import gradio as gr
from unsloth import FastLanguageModel
import torch

# 1. Memuat model yang sudah kamu fine-tune (Ganti nama folder jika berbeda)
print("Sedang memuat model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "qwen-sop-model", # Ini adalah folder hasil download-mu
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
# Mempercepat model untuk inference (wajib dipanggil agar respons chat lebih cepat)
FastLanguageModel.for_inference(model)

# 2. Fungsi logika Chatbot (Versi Anti-Error / Hemat Memori)
def chat_with_ai(message, history):
    # Mengunci PyTorch agar tidak memakan memori untuk training
    with torch.no_grad():
        # Menyusun riwayat percakapan agar model ingat obrolan sebelumnya
        messages = [
            {"role": "system", "content": "Kamu adalah asisten teknisi ahli di pabrik manufaktur. Berikan panduan perbaikan (SOP) langkah demi langkah secara profesional dan utamakan keselamatan (Safety First)."}
        ]

        # Memasukkan riwayat obrolan sebelumnya
        for user_msg, bot_msg in history:
            messages.append({"role": "user", "content": user_msg})
            messages.append({"role": "assistant", "content": bot_msg})

        # Memasukkan pesan terbaru dari pengguna
        messages.append({"role": "user", "content": message})

        # Mengubah format pesan menjadi token
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize = True,
            add_generation_prompt = True,
            return_tensors = "pt",
        ).to("cuda")

        # AI menghasilkan jawaban
        outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)

        # Menerjemahkan token jawaban kembali menjadi teks
        response = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens = True)[0]

        return response

# 3. Membuat Antarmuka (UI) Web
print("Menyiapkan Antarmuka Chatbot...")
demo = gr.ChatInterface(
    fn = chat_with_ai,
    title = "⚙️ Asisten Prescriptive Maintenance AI",
    description = "Masukkan laporan sensor dari Predictive AI, atau tanyakan detail perbaikan mesin kepada asisten.",
    # Baris 'theme' sudah dihapus agar tidak error
    examples = [
        "Laporan Deteksi AI: Mesin terdeteksi mengalami HDF. Data Sensor -> Suhu Udara: 302K, Suhu Proses: 312K, Kecepatan: 1300 rpm, Torsi: 40 Nm, Keausan Alat: 150 menit.",
        "Laporan Deteksi AI: Mesin terdeteksi mengalami HDF dan TWF secara bersamaan. Data Sensor -> Suhu Udara: 300K, Suhu Proses: 312K, Kecepatan: 1350 rpm, Torsi: 45 Nm, Keausan Alat: 215 menit."
    ]
)

# 4. Menjalankan Server Web (Menghasilkan link publik)
demo.launch(share=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Sedang memuat model...
==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.8.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Menyiapkan Antarmuka Chatbot...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1d3c9658140d322bb9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
